In [1]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_np = np.load('Data/X_train_processed.npy')
y_train_np = np.load('Data/y_train_processed.npy')

X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32) #Converting numpy arrays into 32-bit tensors to move the math to the GPU for faster training
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor) #Zipping the inputs and targets together and shuffling them in batches of 64 size so the model learns patterns instead of memorizing the order
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"X_train Tensor Shape: {X_train_tensor.shape}")
print(f"y_train Tensor Shape: {y_train_tensor.shape}")

X_train Tensor Shape: torch.Size([53779, 30, 24])
y_train Tensor Shape: torch.Size([53779])


In [2]:
import torch.nn as nn

class PredictiveMaintenanceBiLSTM(nn.Module):
    def __init__(self, input_size=24, hidden_size=64, num_layers=2):
        super(PredictiveMaintenanceBiLSTM, self).__init__()

        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True) #Bi-LSTM: Reading the 30-cycle history both forwards and backwards

        self.final_decision_layer = nn.Linear(hidden_size * 2, 1) #Creating the funnel to squash the 128 bidirectional thoughts into a single RUL prediction

    def forward(self, x):
        timeline_memory, _ = self.lstm(x)
        final_memory = timeline_memory[:, -1, :] #Slicing the 3D block to grab only the thoughts generated at the very last cycle (cycle 30)
        rul_prediction = self.final_decision_layer(final_memory)
        return rul_prediction

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PredictiveMaintenanceBiLSTM()
model = model.to(device)

print(f"Bi-LSTM Architecture successfully built and loaded onto: {device}")

Bi-LSTM Architecture successfully built and loaded onto: cuda
